In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mrmorj/hate-speech-and-offensive-language-dataset/labeled_data.csv


In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import tensorflow as tf
import numpy as np

2026-06-30 11:52:34.997152: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782820355.245026      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782820355.313767      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782820355.874726      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782820355.874769      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782820355.874773      58 computation_placer.cc:177] computation placer alr

In [3]:
path = '/kaggle/input/datasets/mrmorj/hate-speech-and-offensive-language-dataset/labeled_data.csv'
df = pd.read_csv(path)
df.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


Preprocess

In [4]:
from tensorflow.keras.layers import TextVectorization

In [5]:
X = df['tweet']
y = df['class']

In [6]:
df['class'].astype(int).values

array([2, 1, 1, ..., 1, 1, 2], shape=(24783,))

In [7]:
Max_Feautres = 10000 #Words In The Vocabulary

In [8]:
vectorizer = TextVectorization(max_tokens=Max_Feautres,
                               output_sequence_length=100,
                               output_mode='int')

2026-06-30 11:52:50.876725: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [9]:
vectorizer.adapt(X.values)

In [10]:
vectorized_text = vectorizer(X.values)

In [11]:
dataset = tf.data.Dataset.from_tensor_slices((vectorized_text, y))
dataset = dataset.cache()
dataset = dataset.shuffle(500)
dataset = dataset.batch(16)
dataset = dataset.prefetch(8)

In [12]:
train = dataset.take(int(len(dataset)*0.7))
val = dataset.skip(int(len(dataset)*0.7))
test = dataset.skip(int(len(dataset)*0.9))

Build The Model

In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Bidirectional, Dense, Embedding

In [14]:
model = Sequential()
# Create the embedding layer 
model.add(Embedding(Max_Feautres+1, 64, input_shape=(None,)))
# Bidirectional LSTM Layer
model.add(Bidirectional(LSTM(64, activation='tanh')))
# Feature extractor Fully connected layers
model.add(Dense(128, activation='relu')) 
model.add(Dropout(0.5))                   
model.add(Dense(64, activation='relu'))   
model.add(Dropout(0.5))
# Final layer 
model.add(Dense(3, activation='softmax'))

In [15]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='Adam', metrics=['accuracy'])

In [16]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 64)       │       640,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 731,075 (2.79 MB)

 Trainable params: 731,075 (2.79 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',     
    patience=2,            
    restore_best_weights=True  
)

history = model.fit(
    train,
    epochs=6,
    validation_data=val,
    callbacks=[early_stop],
    verbose=1,
    steps_per_epoch=200
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

Make Predectioins

In [ ]:
input_text = vectorizer([' "Black People Are Stupid!"'])

In [ ]:
res = model.predict(input_text)
(res > 0.5).astype(int)

In [ ]:
batch_X, batch_y = test.as_numpy_iterator().next()

In [ ]:
(model.predict(batch_X) > 0.5).astype(int)

In [ ]:
res.shape

Evaluate Model

In [ ]:
test_loss, test_acc = model.evaluate(test)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

Test The Model

In [ ]:
!pip install gradio jinja2

In [ ]:
import tensorflow as tf
import gradio as gr
from tensorflow.keras.models import load_model

In [ ]:
model.save('TweetsHateSpeech.h5')

In [ ]:
model = tf.keras.models.load_model('TweetsHateSpeech.h5')

In [ ]:
input_str = vectorizer('I hate you!')

In [ ]:
res = model.predict(np.expand_dims(input_str,0))

res

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

# 1. UI Elements Components
text_input = widgets.Textarea(
    value='',
    placeholder='Type your comment/tweet here...',
    description='Comment:',
    layout=widgets.Layout(width='80%', height='100px')
)

output_label = widgets.HTML(value='<b>Result: </b>')
button = widgets.Button(description="Classify", button_style='primary')

# 2. Prediction Function
def classify_comment(b):
    clear_output(wait=True)
    display(text_input, button, output_label)
    
    text = text_input.value
    if text.strip():
        input_vec = vectorizer([text])
        pred = model.predict(input_vec)
        # Target classes aligned with the dataset
        classes = ['Hate Speech', 'Offensive Language', 'Neither']
        result = classes[np.argmax(pred)]
        output_label.value = f'<b>Result:</b> <span style="color:blue;font-size:18px;">{result}</span>'
    else:
        output_label.value = '<b>Result:</b> <span style="color:red;">Please enter some text!</span>'

# 3. Event Listener
button.on_click(classify_comment)

# 4. Display the Interface
display(text_input, button, output_label)